In [1]:
!pip install pandas scikit-learn xgboost joblib

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [2]:
# Load the dataset
df = pd.read_csv('Training.csv')

# The dataset often comes with an empty unnamed column at the end, drop it if it exists
df = df.dropna(axis=1, how='all')

# Separate features (symptoms) and target (disease)
X = df.drop('prognosis', axis=1)
y = df['prognosis']

# Encode the target labels (diseases) into integers for XGBoost
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [3]:
# Initialize and train the XGBoost Classifier
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

print("Training model...")
model.fit(X_train, y_train)
print("Training complete!")

Training model...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:12:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete!


In [4]:
# Make predictions on the test set
predictions = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Decode predictions back to disease names for the report
y_test_decoded = label_encoder.inverse_transform(y_test)
predictions_decoded = label_encoder.inverse_transform(predictions)

print("\nClassification Report:")
print(classification_report(y_test_decoded, predictions_decoded))

Model Accuracy: 100.00%

Classification Report:
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        18
                                   AIDS       1.00      1.00      1.00        30
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        25
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        23
                       Bronchial Asthma       1.00      1.00      1.00        33
                   Cervical spondylosis       1.00      1.00      1.00        23
                            Chicken pox       1.00      1.00      1.00        21
                    Chronic cholestasis       1.00      1.00      1.00        15
                            Common Cold       1.00      1.00

In [5]:
# Create a mapping dictionary
specialty_map = {
    'Fungal infection': 'Dermatologist',
    'Allergy': 'Allergist',
    'GERD': 'Gastroenterologist',
    'Chronic cholestasis': 'Hepatologist',
    'Drug Reaction': 'Allergist/Immunologist',
    'Peptic ulcer diseae': 'Gastroenterologist',
    'AIDS': 'Infectious Disease Specialist',
    'Diabetes ': 'Endocrinologist',
    'Gastroenteritis': 'Gastroenterologist',
    'Bronchial Asthma': 'Pulmonologist',
    'Hypertension ': 'Cardiologist',
    'Migraine': 'Neurologist',
    # ... Add the rest of the 41 diseases from the dataset
}

def predict_symptoms(symptom_vector):
    # Ensure the input is a 2D array
    prediction_encoded = model.predict([symptom_vector])[0]
    disease = label_encoder.inverse_transform([prediction_encoded])[0]
    specialty = specialty_map.get(disease, 'General Physician')

    return {
        "predicted_disease": disease,
        "recommended_specialty": specialty
    }

In [6]:
# Save the model and the label encoder using joblib
joblib.dump(model, 'xgboost_symptom_model.joblib')
joblib.dump(label_encoder, 'label_encoder.joblib')

# Save the feature names (symptoms) so your FastAPI backend knows the exact input array structure
features = list(X.columns)
joblib.dump(features, 'feature_names.joblib')

print("Model, encoder, and features successfully saved!")

Model, encoder, and features successfully saved!
